In [ ]:
import os
import sys
from pathlib import Path
# ── Force deterministic GPU operations for reproducibility ──
# Must be set BEFORE importing JAX so XLA picks it up at init time.
os.environ["XLA_FLAGS"] = os.environ.get("XLA_FLAGS", "") + " --xla_gpu_deterministic_ops=true"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

# Ensure project root is in path
project_root = str(Path.cwd().parent)

if project_root not in sys.path:
    sys.path.append(project_root)

import jax
import jax.numpy as jnp
import numpy as np
import optax
import flax
from flax import linen as nn
import time
import matplotlib.pyplot as plt
import json

from scirex.operators.models.fno import FNO
from scirex.operators.training import create_train_state, TrainState, UnitGaussianNormalizer
from scirex.operators.losses import mse, lp_loss
from scirex.operators.data import random_poisson_batch
from configs.poisson_fno_config import FNO2DConfig
from scirex.operators.losses.physics_eq import pde_loss_FNO
from memory_profiler import profile



def make_schedule(config: FNO2DConfig):
    """Create learning rate schedule with Linear Warmup and Cosine Decay."""
    spe = getattr(config, "steps_per_epoch_actual", config.steps_per_epoch)
    total_steps = config.epochs * spe
    
    # Fixed warmup: ~10 epochs worth of steps
    warmup_steps = min(310, total_steps // 10)
    
    if config.scheduler_type == "cosine":
        cosine_decay_steps = config.cosine_decay_epochs * spe - warmup_steps
        cosine_decay_steps = max(cosine_decay_steps, 1)  # safety
        
        cosine_schedule = optax.cosine_decay_schedule(
            init_value=config.learning_rate,
            decay_steps=cosine_decay_steps,
            alpha=0.0
        )
        schedule = optax.join_schedules(
            schedules=[
                optax.linear_schedule(0.0, config.learning_rate, warmup_steps),
                cosine_schedule
            ],
            boundaries=[warmup_steps]
        )
    elif config.scheduler_type == "step":
        scales = {}
        decay_steps = config.scheduler_step_size * spe
        num_decays = config.epochs // config.scheduler_step_size
        
        current_scale = 1.0
        for i in range(1, num_decays + 1):
            boundary = i * decay_steps
            current_scale *= config.scheduler_gamma
            scales[boundary] = current_scale
            
        schedule = optax.piecewise_constant_schedule(
            init_value=config.learning_rate,
            boundaries_and_scales=scales
        )
    else:
        raise ValueError(f"Unknown scheduler_type: {config.scheduler_type}")
    
    return schedule



# 1. Load Configuration
config = FNO2DConfig()

# Prng Key
rng = jax.random.PRNGKey(config.seed)
rng, init_rng = jax.random.split(rng)

# 2. Initialize Model
# Using config values directly for better accuracy
print(f"Initializing FNO (hidden_channels={config.hidden_channels}, modes={config.n_modes})...")
model = FNO(
    hidden_channels=config.hidden_channels, 
    n_layers=config.n_layers, 
    n_modes=config.n_modes, 
    out_channels=config.out_channels,
    lifting_channel_ratio=config.lifting_channel_ratio,
    projection_channel_ratio=config.projection_channel_ratio,
    use_grid=False, # Data already has grid
    use_norm=config.use_norm, # CRITICAL: Was missing
    fno_skip=config.fno_skip,
    channel_mlp_skip=config.channel_mlp_skip,
    use_channel_mlp=config.use_channel_mlp,
    padding=config.domain_padding,
    activation=nn.gelu
)

# Optimizer & Scheduler
n_train = config.n_train
# Use all data per epoch
steps_per_epoch = n_train // config.batch_size
setattr(config, "steps_per_epoch_actual", steps_per_epoch)

schedule = make_schedule(config)
total_steps = config.epochs * steps_per_epoch

# Data has 3 channels (1 source + 2 grid)
in_channels = 3 
nx, ny = config.resolution
input_shape = (config.batch_size, nx, ny, in_channels)

state = create_train_state(
    rng=init_rng, 
    model=model, 
    input_shape=input_shape, 
    learning_rate=schedule, 
    weight_decay=config.weight_decay
)

# 4. Pre-generate FIXED training and test datasets
n_test = config.n_test
print(f"Generating {n_train} training samples and {n_test} test samples...")
from scirex.operators.data import random_poisson_batch

f_train, u_train = random_poisson_batch(
    batch_size=n_train, nx=nx, ny=ny, channels=1, rng_seed=config.seed
)
f_test, u_test = random_poisson_batch(
    batch_size=n_test, nx=nx, ny=ny, channels=1, rng_seed=999
)

f_train = jnp.asarray(f_train)
u_train = jnp.asarray(u_train)
f_test = jnp.asarray(f_test)
u_test = jnp.asarray(u_test)

# 5. Normalize Data globally
x_normalizer = UnitGaussianNormalizer(f_train)
y_normalizer = UnitGaussianNormalizer(u_train)

f_train_encoded = x_normalizer.encode(f_train)
u_train_encoded = y_normalizer.encode(u_train)
f_test_encoded = x_normalizer.encode(f_test)

test_batch_encoded = {"x": f_test_encoded, "y": y_normalizer.encode(u_test)}
print(f"Data shapes: f_train={f_train.shape}, u_train={u_train.shape}")

# Paths
ckpt_dir = os.path.join(project_root, "experiments/checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_path = os.path.join(ckpt_dir, "poisson2d_fno_params.pkl")

results_dir = os.path.join(project_root, "experiments/results/poisson2d_fno")
os.makedirs(results_dir, exist_ok=True)
rng_key = jax.random.PRNGKey(config.seed + 1)
best_rel_l2 = float("inf")
history = {"train_rel_l2": [], "test_rel_l2": []}
rng_key, shuffle_key = jax.random.split(rng_key)
perm = jax.random.permutation(shuffle_key, n_train)
f_shuffled = f_train_encoded[perm]
u_shuffled = u_train_encoded[perm]
# 5. Training Step (Optimized for Gradient Stability)






for epoch in range(config.epochs):
    epoch_start_time = time.time()
    epoch_loss = 0.0
    
    # Shuffle training data each epoch
    rng_key, shuffle_key = jax.random.split(rng_key)
    perm = jax.random.permutation(shuffle_key, n_train)
    f_shuffled = f_train_encoded[perm]
    u_shuffled = u_train_encoded[perm]
    
    for step in range(steps_per_epoch):
        start_idx = step * config.batch_size
        end_idx = start_idx + config.batch_size
        batch = {"x": f_shuffled[start_idx:end_idx], "y": u_shuffled[start_idx:end_idx]}
        pred_encoded, intermediate_output = state.apply_fn({"params": state.params}, batch["x"], mutable=["intermediates"])
        print(intermediate_output["intermediates"]["last"][0].shape)


Initializing FNO (hidden_channels=128, modes=(24, 24))...
After FNO Block 1, shape: (32, 64, 64, 128)
After FNO Block 2, shape: (32, 64, 64, 128)
After FNO Block 3, shape: (32, 64, 64, 128)
After FNO Block 4, shape: (32, 64, 64, 128)
Generating 2000 training samples and 200 test samples...
Data shapes: f_train=(2000, 64, 64, 3), u_train=(2000, 64, 64, 1)
After FNO Block 1, shape: (32, 64, 64, 128)
After FNO Block 2, shape: (32, 64, 64, 128)
After FNO Block 3, shape: (32, 64, 64, 128)
After FNO Block 4, shape: (32, 64, 64, 128)
(32, 64, 64, 128)
After FNO Block 1, shape: (32, 64, 64, 128)
After FNO Block 2, shape: (32, 64, 64, 128)
After FNO Block 3, shape: (32, 64, 64, 128)
After FNO Block 4, shape: (32, 64, 64, 128)
(32, 64, 64, 128)
After FNO Block 1, shape: (32, 64, 64, 128)
After FNO Block 2, shape: (32, 64, 64, 128)
After FNO Block 3, shape: (32, 64, 64, 128)
After FNO Block 4, shape: (32, 64, 64, 128)
(32, 64, 64, 128)
After FNO Block 1, shape: (32, 64, 64, 128)
After FNO Block 2

KeyboardInterrupt: 

In [ ]:
import scipy.io
import jax.numpy as jnp
import os
import sys
import zenodo_get 
from dataclasses import dataclass, field
from pathlib import Path
extension = "ones" # Options: "broadcast", "zeros", "ones"

# Assuming project_root was defined as a string earlier, wrap it in Path()
project_root = Path(project_root) 

# Sys.path.append usually expects a string, so you might need to cast it back
sys.path.append(str(project_root))

# Now the '/' operator will work perfectly
data_path = project_root / 'data' / 'burgers_v100_t100_r1024_N2048.mat'
data = scipy.io.loadmat(data_path)

input_function = data['input']   #<---- This input function is the field u at t=0 at x=(0,pi)
output_function = jnp.expand_dims(data['output'],axis=3) #<---- This output function is the field u at t=linspace(0,1,100) for the same x grid as input function


resolution = len(data['input'][0])#<---- This is the resolution of the grid on which the field u is defined. It is 1024 for this dataset.
in_channels = 2 # 1 for u and 1 for x

batch_size = len(input_function)

nt = 101  # 1 step for t=0, plus 100 future steps
in_channels_new = 3 # x, t, u

# 1. Construct the complete (x, t) meshgrid for the entire space-time domain
x_grid_array = jnp.linspace(0, jnp.pi, resolution)
t_grid_array = jnp.linspace(0, 1, nt) 

# 'ij' indexing ensures the mesh grid aligns with shape (nt, resolution) -> (101, 1024)
T_mesh, X_mesh = jnp.meshgrid(t_grid_array, x_grid_array, indexing='ij')

# Broadcast the meshgrids to include the batch dimension -> (2048, 101, 1024)
X_grid = jnp.broadcast_to(X_mesh, (batch_size, nt, resolution))
T_grid = jnp.broadcast_to(T_mesh, (batch_size, nt, resolution))

# 2. Isolate the 'u' channel and apply the extension methods ONLY to 'u'
# Initial condition shape: (2048, 1024). Expand to: (2048, 1, 1024)
u0_expanded = jnp.expand_dims(jnp.array(input_function), axis=1)

if extension == "broadcast":
    # Broadcast u(t=0) uniformly across all 101 time steps
    U_grid = jnp.broadcast_to(u0_expanded, (batch_size, nt, resolution))
    
elif extension == "zeros":
    # Keep true values at u(t=0), but pad all t>0 steps with zeros
    zeros_u = jnp.zeros((batch_size, nt - 1, resolution))
    U_grid = jnp.concatenate([u0_expanded, zeros_u], axis=1)
    
elif extension == "ones":
    # Keep true values at u(t=0), but pad all t>0 steps with ones
    ones_u = jnp.ones((batch_size, nt - 1, resolution))
    U_grid = jnp.concatenate([u0_expanded, ones_u], axis=1)
    
else:
    raise ValueError("Invalid extension method. Choose from 'broadcast', 'zeros', or 'ones'.")

# 3. Stack x, t, and u together along the final channel axis
# Final shape: (batch_size, nt, resolution, 3) -> (2048, 101, 1024, 3)
input_data_3D = jnp.stack([X_grid, T_grid, U_grid], axis=-1)


/home/zenteiq/Documents/SciREX/.venv/lib/python3.12/site-packages/scipy/io/matlab/_mio.py:236: MatReadWarning: Duplicate variable name "None" in stream - replacing previous with new
Considerscipy.io.matlab.varmats_from_mat to split file into single variable files
  matfile_dict = MR.get_variables(variable_names)


In [29]:
print(input_data_3D.shape)
print(input_data_3D[0,0,4,:])

(2048, 101, 1024, 3)
[ 0.01228384  0.         -0.62198544]


In [40]:
print(output_function[5,100,1023])

0.019896988509758996


In [48]:
actual_output=jnp.zeros((2048,101,1024,1))
for batch in range(2048):
    for nt in range(101):
        for nx in range(1024):
            actual_output.at[(batch,nt,nx,1)].set(output_function[batch,nt,nx])

KeyboardInterrupt: 

In [67]:
actual_output=jnp.expand_dims(jnp.array(output_function), axis=3)

In [93]:
print(actual_output.shape)
print(actual_output[2000,90,1023,1])

(2048, 101, 1024, 1)
-0.086925


In [ ]:
import jax.numpy as jnp
input 

In [1]:
import numpy as np

# Create 1D axes
x = np.linspace(0, 5, 6)
y = np.linspace(0, 3, 4)

# Create the 2D grids
X, Y = np.meshgrid(x, y)

print("X grid:\n", X)
print("\nY grid:\n", Y)

X grid:
 [[0. 1. 2. 3. 4. 5.]
 [0. 1. 2. 3. 4. 5.]
 [0. 1. 2. 3. 4. 5.]
 [0. 1. 2. 3. 4. 5.]]

Y grid:
 [[0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1.]
 [2. 2. 2. 2. 2. 2.]
 [3. 3. 3. 3. 3. 3.]]


In [2]:
with open('20250804_stator_magnetOD_28_1_Az_30d.txt', 'r') as file:
    data = file.read()
    print(data)

x	y	z	Ax	Ay	Az
2.4195905753766699e-02	-1.6244952060724401e-02	0.0000000000000000e+00	0.0000000000000000e+00	0.0000000000000000e+00	-1.0550631210207939e-02
2.4720448839103699e-02	-1.5915779822625500e-02	0.0000000000000000e+00	0.0000000000000000e+00	0.0000000000000000e+00	-1.0222904384136200e-02
2.4738650583924202e-02	-1.6560780692279051e-02	0.0000000000000000e+00	0.0000000000000000e+00	0.0000000000000000e+00	-1.0558299720287325e-02
2.5244991924440699e-02	-1.5586607584526600e-02	0.0000000000000000e+00	0.0000000000000000e+00	0.0000000000000000e+00	-9.8880277946591377e-03
2.5263193669261198e-02	-1.6231608454180150e-02	0.0000000000000000e+00	0.0000000000000000e+00	0.0000000000000000e+00	-1.0209649801254274e-02
2.5281395414081701e-02	-1.6876609323833700e-02	0.0000000000000000e+00	0.0000000000000000e+00	0.0000000000000000e+00	-1.0564409196376801e-02
2.6166494044731099e-02	-1.2831793019973800e-02	0.0000000000000000e+00	0.0000000000000000e+00	0.0000000000000000e+00	-8.5332626476883888e-03
2.671

In [3]:
del data

In [16]:
import pandas as pd
import numpy as np

# 1. Load data
df = pd.read_csv('20250804_stator_magnetOD_28_1_Az_30d.txt', sep='\s+')

# 2. Force conversion to numbers
df['x'] = pd.to_numeric(df['x'], errors='coerce')
df['y'] = pd.to_numeric(df['y'], errors='coerce')

# 3. Clean up any rows that aren't numbers (like headers)
df = df.dropna().reset_index(drop=True)

# 4. Perform calculations
df['r'] = np.sqrt(df['x']**2 + df['y']**2)
df['theta'] = np.arctan2(df['y'], df['x'])

print(df.head())

          x         y    z   Ax   Ay        Az         r     theta
0  0.024196 -0.016245  0.0  0.0  0.0 -0.010551  0.029143 -0.591267
1  0.024720 -0.015916  0.0  0.0  0.0 -0.010223  0.029401 -0.572026
2  0.024739 -0.016561  0.0  0.0  0.0 -0.010558  0.029770 -0.589913
3  0.025245 -0.015587  0.0  0.0  0.0 -0.009888  0.029669 -0.553125
4  0.025263 -0.016232  0.0  0.0  0.0 -0.010210  0.030028 -0.571085


<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_383518/2578730848.py:5: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv('20250804_stator_magnetOD_28_1_Az_30d.txt', sep='\s+')


In [29]:
# Sort primarily by 'r' (ascending), then by 'theta' (ascending)
df.sort_values(by=['x','y'], ascending=[True,True], inplace=True)

print(df.head())

         x         y    z   Ax   Ay        Az         r     theta
24165  0.0 -0.041018  0.0  0.0  0.0  0.000000  0.041018 -1.570796
24072  0.0 -0.041018  0.0  0.0  0.0  0.000000  0.041018 -1.570796
24073  0.0 -0.040557  0.0  0.0  0.0 -0.000382  0.040557 -1.570796
24372  0.0 -0.040095  0.0  0.0  0.0 -0.000775  0.040095 -1.570796
24371  0.0 -0.040095  0.0  0.0  0.0 -0.000775  0.040095 -1.570796


In [31]:
df['x_diff'] = df['x'].diff()
df['y_diff'] = df['y'].diff()
print(df.head())

         x         y    z   Ax   Ay        Az         r     theta  x_diff  \
24165  0.0 -0.041018  0.0  0.0  0.0  0.000000  0.041018 -1.570796     NaN   
24072  0.0 -0.041018  0.0  0.0  0.0  0.000000  0.041018 -1.570796     0.0   
24073  0.0 -0.040557  0.0  0.0  0.0 -0.000382  0.040557 -1.570796     0.0   
24372  0.0 -0.040095  0.0  0.0  0.0 -0.000775  0.040095 -1.570796     0.0   
24371  0.0 -0.040095  0.0  0.0  0.0 -0.000775  0.040095 -1.570796     0.0   

         y_diff  
24165       NaN  
24072  0.000000  
24073  0.000462  
24372  0.000462  
24371  0.000000  


In [33]:
print(df['x_diff'].unique().tolist())

[nan, 0.0, 0.00020434970330271047, 2.4936649967166602e-18, 1.4625837897010039e-05, 3.469446951953614e-18, 2.5493570540587507e-05, 1.0299920638612292e-18, 1.8544246302299307e-06, 1.0842021724855044e-18, 3.022299934100003e-06, 5.4752209710517974e-18, 1.0261037358648526e-05, 1.463672932855431e-18, 4.3714106701611016e-05, 1.6652459738289514e-05, 6.309005411168651e-05, 5.5294310796760726e-18, 1.9741703437241493e-05, 5.894208953274491e-06, 4.336808689942018e-19, 5.041540102057596e-18, 2.9251675794019535e-05, 4.87890977618477e-19, 6.5052130349130266e-18, 8.919284147748598e-06, 2.439454888092385e-18, 1.5935641517676508e-05, 2.6132215415741993e-05, 2.0599841277224584e-18, 2.0014065919099445e-06, 1.7074426685475316e-06, 6.0445998681995725e-06, 1.0950441942103595e-17, 7.636506200517043e-06, 4.87890977618477e-18, 1.288556851677513e-05, 2.927345865710862e-18, 1.951563910473908e-18, 1.279868265318502e-05, 5.095750210681871e-18, 1.949160644558988e-05, 1.0083080204115191e-17, 5.513792430443e-05, 6.407

In [35]:
df.sort_values(by=['r','theta'], ascending=[True,True], inplace=True)
df['delta_r'] = df['r'].diff()
df['delta_theta'] = df['theta'].diff()
print(df['delta_theta'].unique().tolist())

[nan, 0.243690590127422, 1.3271057366674746, -2.3380856923935247, 1.2908881410698343, 0.2799081857250623, 1.2908881410698345, -1.8144869168089945, 1.5707963267948966, -2.3743032878365353, 1.0980899484596018, -2.9634692626724615, 1.3926729358775645, 0.17812339091733276, 1.3672267371750881, -2.9125768652675093, 1.341780538472612, 0.2290157883222848, -1.6725811216048012, 0.4981525768958224, 0.5490449743007759, 0.025446198702476087, 1.0471975511965983, -3.0652540574823655, 0.4727063781933465, 0.05089239740495033, 0.39636778208592305, 0.4727063781933464, 0.5744911730032507, 0.025446198702475775, 0.47270637819334693, -2.4653166857766395, 0.3454753846809683, 0.6762759678131546, 0.5490449743007738, 0.17812339091733254, 0.025446198702475886, 0.37092158338344206, 1.1235361473040266, 0.37092158338344194, 0.07633859610742824, -1.1998747434114547, 0.1272309935123792, 0.44726017949087415, 0.12723099351237904, 0.8436279615767883, 0.025446198702475997, 0.07633859610742766, -2.2216260959055774, 0.02544